# Ejemplo de uso de DataFrames

* La SparkSession ya viene creada en la variable spark 
* Utilizamos el SparkContext que hay dentro de la SparkSession. El SparkContext era en versiones primitivas de Spark el equivalente a la SparkSession actual

In [4]:
flightsRawDF = spark.read.option("header", "true")\
                     .option("inferSchema", "true")\
                     .csv("gs://unirbucket2023/datos/flights.csv")  # modifica la ruta para poner tu bucket de GCS

In [5]:
flightsRawDF.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- dep_time: string (nullable = true)
 |-- dep_delay: string (nullable = true)
 |-- arr_time: string (nullable = true)
 |-- arr_delay: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- air_time: string (nullable = true)
 |-- distance: integer (nullable = true)
 |-- hour: string (nullable = true)
 |-- minute: string (nullable = true)



In [7]:
flightsRawDF.show(10)

+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute|
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|2014|    1|  1|       1|       96|     235|       70|     AS| N508AS|   145|   PDX| ANC|     194|    1542|   0|     1|
|2014|    1|  1|       4|       -6|     738|      -23|     US| N195UW|  1830|   SEA| CLT|     252|    2279|   0|     4|
|2014|    1|  1|       8|       13|     548|       -4|     UA| N37422|  1609|   PDX| IAH|     201|    1825|   0|     8|
|2014|    1|  1|      28|       -2|     800|      -23|     US| N547UW|   466|   PDX| CLT|     251|    2282|   0|    28|
|2014|    1|  1|      34|       44|     325|       43|     AS| N762AS|   121|   SEA| ANC|     201|    1448|   0|    34|
|2014|    1|  1|      37|       82|     

In [10]:
n_filas = flightsRawDF.count()

In [11]:
n_filas

162049

In [12]:
flightsRawDF.is_cached

False

In [13]:
flightsRawDF.count()

162049

In [14]:
flightsRawDF.cache()

DataFrame[year: int, month: int, day: int, dep_time: string, dep_delay: string, arr_time: string, arr_delay: string, carrier: string, tailnum: string, flight: int, origin: string, dest: string, air_time: string, distance: int, hour: string, minute: string]

In [15]:
flightsRawDF.is_cached

True

In [16]:
flightsRawDF.count()

162049

In [18]:
flightsRawDF.show()

+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute|
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|2014|    1|  1|       1|       96|     235|       70|     AS| N508AS|   145|   PDX| ANC|     194|    1542|   0|     1|
|2014|    1|  1|       4|       -6|     738|      -23|     US| N195UW|  1830|   SEA| CLT|     252|    2279|   0|     4|
|2014|    1|  1|       8|       13|     548|       -4|     UA| N37422|  1609|   PDX| IAH|     201|    1825|   0|     8|
|2014|    1|  1|      28|       -2|     800|      -23|     US| N547UW|   466|   PDX| CLT|     251|    2282|   0|    28|
|2014|    1|  1|      34|       44|     325|       43|     AS| N762AS|   121|   SEA| ANC|     201|    1448|   0|    34|
|2014|    1|  1|      37|       82|     

In [21]:
flightsRawDF.unpersist()

DataFrame[year: int, month: int, day: int, dep_time: string, dep_delay: string, arr_time: string, arr_delay: string, carrier: string, tailnum: string, flight: int, origin: string, dest: string, air_time: string, distance: int, hour: string, minute: string]

In [22]:
flightsRawDF.is_cached

False

### No ha reconocido correctamente el tipo de `arr_delay` por la presencia de `NA` que es un string normal para Spark

Contamos las filas que tienen un NA en `arr_delay`

In [19]:
from pyspark.sql import functions as F

filtradoDF = flightsRawDF.where("arr_delay = 'NA'")

In [21]:
cuantos_arrdelay_na = filtradoDF.count()

In [22]:
cuantos_arrdelay_na

1301

In [23]:
filtradoDF.is_cached

False

In [28]:
filtradoDF.cache()

DataFrame[year: int, month: int, day: int, dep_time: string, dep_delay: string, arr_time: string, arr_delay: string, carrier: string, tailnum: string, flight: int, origin: string, dest: string, air_time: string, distance: int, hour: string, minute: string]

In [20]:
filtradoDF.unpersist()

DataFrame[year: int, month: int, day: int, dep_time: string, dep_delay: string, arr_time: string, arr_delay: string, carrier: string, tailnum: string, flight: int, origin: string, dest: string, air_time: string, distance: int, hour: string, minute: string]

In [24]:
from pyspark.sql import types as T
from pyspark.sql import functions as F


# Reemplazamos la columna arr_delay por el resultado de convertirla a entero
flightsDF = flightsRawDF.select(F.col("arr_delay"),
                                F.col("origin"), 
                                "dest", "year", "carrier",
                                F.sqrt(F.col("arr_delay")/60).alias("arr_delay_horas")
                               )\
                        .where("arr_delay != 'NA'")\
                        .withColumn("arr_delay", F.col("arr_delay").cast("int"))\
                        .drop("carrier")\
                        .withColumnRenamed("year", "anio")\
                        .cache()

flightsDF.show()

In [25]:
flightsDF.printSchema()

root
 |-- arr_delay: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- arr_delay_horas: double (nullable = true)



In [23]:
flightsDF.cache() # porque lo voy a usar en varias operaciones de ramas distintas

25/05/11 08:42:42 WARN CacheManager: Asked to cache already cached data.


DataFrame[arr_delay: int, origin: string, dest: string, anio: int, arr_delay_horas: double]

In [24]:
algunasColsDF = flightsDF.select((F.col("arr_delay") / 60).alias("arr_delay_horas"),
                                 "origin", "dest")

algunasColsDF.show()

+--------------------+------+----+
|     arr_delay_horas|origin|dest|
+--------------------+------+----+
|  1.1666666666666667|   PDX| ANC|
|-0.38333333333333336|   SEA| CLT|
|-0.06666666666666667|   PDX| IAH|
|-0.38333333333333336|   PDX| CLT|
|  0.7166666666666667|   SEA| ANC|
|  1.4666666666666666|   SEA| DTW|
|                3.65|   SEA| ORD|
|                0.25|   PDX| IAH|
|                 0.4|   SEA| DEN|
|                -0.1|   SEA| EWR|
| 0.06666666666666667|   PDX| DEN|
|                 0.2|   PDX| PHX|
|                -0.2|   SEA| SLC|
|-0.26666666666666666|   SEA| DFW|
| -0.4166666666666667|   SEA| ANC|
|-0.03333333333333333|   SEA| SJC|
|               -0.15|   PDX| DEN|
|-0.31666666666666665|   SEA| ORD|
|-0.13333333333333333|   SEA| LAX|
| 0.08333333333333333|   SEA| DEN|
+--------------------+------+----+
only showing top 20 rows



In [25]:
lista = algunasColsDF.take(4)
lista

[Row(arr_delay_horas=1.1666666666666667, origin='PDX', dest='ANC'),
 Row(arr_delay_horas=-0.38333333333333336, origin='SEA', dest='CLT'),
 Row(arr_delay_horas=-0.06666666666666667, origin='PDX', dest='IAH'),
 Row(arr_delay_horas=-0.38333333333333336, origin='PDX', dest='CLT')]

In [26]:
algunasColsDF.rdd.take(4)

[Row(arr_delay_horas=1.1666666666666667, origin='PDX', dest='ANC'),
 Row(arr_delay_horas=-0.38333333333333336, origin='SEA', dest='CLT'),
 Row(arr_delay_horas=-0.06666666666666667, origin='PDX', dest='IAH'),
 Row(arr_delay_horas=-0.38333333333333336, origin='PDX', dest='CLT')]

### Función when dentro de withColumn para renombrar niveles o reemplazar valores

Truncamos los retrasos negativos (vuelos que llegaron con adelanto) para que sea 0, y cambiamos "SEA" por "Seattle" en el origin cuando el retraso sea 0.

In [27]:
otroDF = algunasColsDF.withColumn("arr_delay", F.when(F.col("arr_delay_horas") < 0, 0)\
                                                .otherwise(F.col("arr_delay_horas")))\
                      .withColumn("origin", F.when( (F.col("arr_delay") == 0) & (F.col("origin") == "SEA"), "Seattle")\
                                             .otherwise(F.col("origin")))

In [28]:
otroDF.show()

+--------------------+-------+----+-------------------+
|     arr_delay_horas| origin|dest|          arr_delay|
+--------------------+-------+----+-------------------+
|  1.1666666666666667|    PDX| ANC| 1.1666666666666667|
|-0.38333333333333336|Seattle| CLT|                0.0|
|-0.06666666666666667|    PDX| IAH|                0.0|
|-0.38333333333333336|    PDX| CLT|                0.0|
|  0.7166666666666667|    SEA| ANC| 0.7166666666666667|
|  1.4666666666666666|    SEA| DTW| 1.4666666666666666|
|                3.65|    SEA| ORD|               3.65|
|                0.25|    PDX| IAH|               0.25|
|                 0.4|    SEA| DEN|                0.4|
|                -0.1|Seattle| EWR|                0.0|
| 0.06666666666666667|    PDX| DEN|0.06666666666666667|
|                 0.2|    PDX| PHX|                0.2|
|                -0.2|Seattle| SLC|                0.0|
|-0.26666666666666666|Seattle| DFW|                0.0|
| -0.4166666666666667|Seattle| ANC|             

In [29]:
agregacionesDF = flightsDF.groupBy("origin", "dest")\
                          .agg(F.mean("arr_delay").alias("avg_arr_delay"),
                               F.max("arr_delay").alias("max_arr_delay")                              
                              )

agregacionesDF.printSchema()

root
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- avg_arr_delay: double (nullable = true)
 |-- max_arr_delay: integer (nullable = true)

